In [ ]:
# ==========================================
# 3_visualization_inference.ipynb
# ==========================================

import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm


# ------------------------------
# Load the trained generator
# ------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'

generator = AttentionUNet(in_channels=1, out_channels=1).to(device)
generator.load_state_dict(torch.load('generator_wt.pt', map_location=device))
generator.eval()

# ------------------------------
# Inference on Validation Set
# ------------------------------
with torch.inference_mode():
    images, labels, masked_images = next(iter(dataloaders['val']))
    images = images.to(device)
    masked_images = masked_images.to(device)
    
    reconstructions = generator(masked_images)
    
    # Move back to CPU for visualization
    images = images.cpu()
    masked_images = masked_images.cpu()
    reconstructions = reconstructions.cpu()

    # De-normalize
    images = invTrans(images)
    masked_images = invTrans(masked_images)
    reconstructions = invTrans(reconstructions)

# ------------------------------
# Visualization function
# ------------------------------
def visualize_result(orig, masked, recon):
    n = orig.shape[0]
    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))

    for i in range(n):
        # Original
        axes[i, 0].imshow(orig[i].permute(1,2,0), cmap='gray')
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')

        # Masked
        axes[i, 1].imshow(masked[i].permute(1,2,0), cmap='gray')
        axes[i, 1].set_title('Masked')
        axes[i, 1].axis('off')

        # Reconstructed
        axes[i, 2].imshow(recon[i].permute(1,2,0), cmap='gray')
        axes[i, 2].set_title('Reconstructed')
        axes[i, 2].axis('off')

        # Difference Heatmap
        diff = torch.abs(orig[i] - recon[i])
        diff_img = transforms.functional.to_pil_image(diff)
        diff_img = transforms.functional.to_grayscale(diff_img, num_output_channels=1)
        axes[i, 3].imshow(diff_img, cmap='jet')
        axes[i, 3].set_title('Difference Heatmap')
        axes[i, 3].axis('off')

    plt.show()

# ------------------------------
# Run visualization
# ------------------------------
visualize_result(images, masked_images, reconstructions)
